# Notebook 05 · Pilot experiment: RetinaNet versus YOLO on one fold

The training and evaluation code is not in this notebook: it is in the `ichdet` package (`src/ichdet/` in
this repository).

**Before running:**

1. Notebook 04 already run (it writes `tc/preparo/cq500_512.zip` to the Drive).
2. Copy the folder `src/ichdet` of this repository to the Drive as `MyDrive/ich-benchmark/codigo/ichdet/`
   (replacing any older copy). Cell 1 prints a fingerprint of the code, so that the version that ran is
   known.
3. **A GPU is needed:** *Runtime → Change runtime type → GPU* (L4 or A100).

**What the pilot does:** trains RetinaNet and YOLOv8m without mosaic on folds 2 to 5 of CQ500 and validates
on fold 1, with few epochs (5). The single evaluator then compares the two.

**Purpose (fixed in the statistical plan, `docs/statistical_protocol.md`):** (a) confirm that the whole
pipeline works; (b) measure the GPU time of each training job, to estimate the full experiment; (c) estimate
the variability of the difference between detectors, to project the width of the final interval. It is
**not** used to choose hyperparameters or to draw conclusions about which detector is better. The test set
(CT-ICH) is **not** used here.

**If Colab disconnects:** run everything again. Training resumes from the last epoch saved on the Drive.

**Language:** documentation and comments are in English; identifiers, printed messages and the column
names of the files written here are in Portuguese, exactly as in the run that produced the results.

In [ ]:
# 1. Drive, libraries, GPU and code version
import hashlib, json, os, subprocess, sys, time, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/ich-benchmark')
    LOCAL = Path('/content/cq500_512')               # Colab disk: reading many images from the Drive is slow
    # MONAI 1.5.2 accepts numpy 2. (1.4.0 requires numpy < 2 and, by downgrading Colab's numpy, breaks the
    # libraries already installed: "numpy.dtype size changed".) The detection code is the same as in 1.4.0.
    from importlib.metadata import version
    numpy_antes = version('numpy')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'monai==1.5.2', 'ultralytics>=8.3,<8.4'],
                   check=True)
    if version('numpy') != numpy_antes:
        raise SystemExit(f'O pip trocou o numpy ({numpy_antes} -> {version("numpy")}). '
                         'Reinicie a sessão (Ambiente de execução → Reiniciar sessão) e rode de novo.')
except ImportError:                                  # local test: RAIZ mimics the Drive folder
    RAIZ = Path(os.environ['PILOTO_LOCAL'])
    LOCAL = RAIZ / 'cq500_512_local'

CODIGO = RAIZ / 'codigo'
sys.path.insert(0, str(CODIGO))
PREPARO = RAIZ / 'dados' / 'tc' / 'preparo'
SAIDA = RAIZ / 'dados' / 'tc' / 'experimento' / 'piloto'
SAIDA.mkdir(parents=True, exist_ok=True)

import numpy as np, torch, monai, ultralytics
import ichdet
from ichdet import retinanet, yolo, pilot, training_config as cfg
from ichdet.data_io import load_monai_split, save_predictions
from ichdet.evaluate import compare_crossval

def impressao_digital(pasta):
    h = hashlib.sha256()
    for p in sorted(Path(pasta).rglob('*.py')):
        h.update(p.name.encode()); h.update(p.read_bytes())
    return h.hexdigest()[:16]

AMBIENTE = {
    'ichdet_version': ichdet.__version__, 'ichdet_fingerprint': impressao_digital(CODIGO / 'ichdet'),
    'python': sys.version.split()[0], 'numpy': np.__version__, 'torch': torch.__version__, 'monai': monai.__version__,
    'ultralytics': ultralytics.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
    'date': time.strftime('%Y-%m-%d'),
}
print(json.dumps(AMBIENTE, indent=2))
assert torch.cuda.is_available() or 'PILOTO_LOCAL' in os.environ, 'Ative a GPU (Ambiente de execução → Alterar o tipo).'

In [ ]:
# 2. CQ500 images on the Colab disk and the lists of fold 1 (training: folds 2-5; validation: fold 1)
if not (LOCAL / 'monai' / 'fold1.json').exists():
    inicio = time.time()
    with zipfile.ZipFile(PREPARO / 'cq500_512.zip') as z:
        z.extractall(LOCAL)
    print(f'descompactado em {time.time() - inicio:.0f} s')

dividido = json.loads((LOCAL / 'monai' / 'fold1.json').read_text())
TREINO, VALIDACAO = dividido['training'], dividido['validation']
resumo_export = json.loads((PREPARO / 'export_512_summary.json').read_text())
esperado = resumo_export['cq500_folds_main_analysis']['fold1']
exames_val = {e['exam'] for e in VALIDACAO}
print(f'treino: {len(TREINO)} imagens; validação: {len(VALIDACAO)} imagens de {len(exames_val)} exames')
assert len(VALIDACAO) == esperado['slices'] and len(exames_val) == esperado['exams'], 'listas diferentes do notebook 04'
assert not exames_val & {e['exam'] for e in TREINO}, 'exame na validação e no treino ao mesmo tempo'
GT = load_monai_split(LOCAL / 'monai' / 'fold1.json', 'validation')

In [ ]:
# 3. RetinaNet (MONAI): training for a fixed number of epochs, last epoch kept, and prediction on fold 1
pasta_r = SAIDA / 'retinanet'
ponto = retinanet.train(TREINO, LOCAL, pasta_r, epochs=cfg.PILOT_EPOCHS)
inicio = time.time()
PRED_R = retinanet.predict(ponto, VALIDACAO, LOCAL)
seg_pred_r = time.time() - inicio
save_predictions(SAIDA / 'pred_retinanet_fold1.csv', PRED_R)
hist_r = json.loads((pasta_r / 'history.json').read_text())
print(f'predição: {seg_pred_r:.0f} s; {sum(len(s) for _, s in PRED_R.values())} caixas')

In [ ]:
# 4. YOLOv8m (Ultralytics) without mosaic: same images, epochs, optimiser and flipping
dados_yolo = yolo.write_dataset(TREINO, LOCAL, SAIDA / 'yolo_dados')
pesos = yolo.train(dados_yolo, SAIDA / 'yolo', 'sem_mosaico', epochs=cfg.PILOT_EPOCHS, mosaic=False)
inicio = time.time()
PRED_Y = yolo.predict(pesos, VALIDACAO, LOCAL)
seg_pred_y = time.time() - inicio
save_predictions(SAIDA / 'pred_yolo_sem_mosaico_fold1.csv', PRED_Y)
print(f'predição: {seg_pred_y:.0f} s; {sum(len(s) for _, s in PRED_Y.values())} caixas')

In [ ]:
# 5. Single evaluator, estimate of the full experiment and summary (in English)
import csv

def segundos_por_epoca_yolo(pasta):
    # Ultralytics results.csv: column "time" = cumulative seconds at the end of each epoch
    linhas = list(csv.DictReader(open(pasta / 'results.csv')))
    tempos = [float(l['time']) for l in linhas]
    return float(np.median(np.diff([0.0] + tempos)[1:])) if len(tempos) > 1 else tempos[0]

comparacao = compare_crossval([(GT, PRED_R, PRED_Y)])
seg_epoca = {'retinanet': float(np.median([h['seconds'] for h in hist_r[1:]] or [hist_r[0]['seconds']])),
             'yolo_no_mosaic': segundos_por_epoca_yolo(SAIDA / 'yolo' / 'sem_mosaico')}
resumo = {
    'environment': AMBIENTE,
    'design': 'pilot: train on CQ500/BHX folds 2-5, validate on fold 1; CT-ICH not used',
    'training_images': len(TREINO), 'validation_images': len(VALIDACAO), 'validation_exams': len(exames_val),
    'retinanet_loss_per_epoch': [round(h['loss'], 4) for h in hist_r],
    'prediction_seconds': {'retinanet': round(seg_pred_r), 'yolo_no_mosaic': round(seg_pred_y)},
    'comparison_A_retinanet_B_yolo_no_mosaic': comparacao,
    **pilot.summarize(comparacao, seg_epoca, pilot_exams=len(exames_val)),
}
(SAIDA / 'pilot_summary.json').write_text(json.dumps(resumo, indent=2, default=float))
print(json.dumps({k: resumo[k] for k in ('seconds_per_epoch', 'estimated_training_gpu_hours_full_experiment',
                                         'precision_projection')}, indent=2, default=float))
print('AP@0.5 (piloto, poucas épocas) — RetinaNet:', round(comparacao['A']['ap50'], 3),
      '| YOLO sem mosaico:', round(comparacao['B']['ap50'], 3))

## After running

1. `dados/tc/experimento/piloto/pilot_summary.json` on the Drive records the pilot (it is included in this
   repository under `data/`); predictions and weights stay on the Drive.
2. The AP of the pilot is **not** a result of the study: the models were trained for only 5 epochs.